In [ ]:
# Step 1: Check Python vresion
import sys
print(f"Pythonn version: {sys.version}")

# Check if Python version is 3.9 or higher
if sys.version_info >= (3,9):
  print("Python version is compatible.")
else:
  print("Please upgrade to Python 3.9 or higher.")

In [ ]:
# Step 2: Import Langchain and check version
import langchain
from langchain_core import __version__ as core_version

print(f"Langchain version: {langchain.__version__}")
print(f"Langchain core version: {core_version}")

# We are using Langchain version 1.0.5+ for this course
if langchain.__version__ >= "1.0.5":
  print("Langchain version is compatible.")
else:
  print("Please upgrade to Langchain version 1.0.5 or higher.")

In [ ]:
# Step: 3: Load environment variables (API keys)
from dotenv import load_dotenv
import os

load_dotenv()

# Check if OpenAI API key is set
if os.getenv("OPENAI_API_KEY"):
  print("OpenAI API key is set.")
  
else:
  print("Please set the OPENAI_API_KEY environment variable.")

# Check for Google API key
if os.getenv("GOOGLE_API_KEY"):
  print("Google API key is set.")
else:
  print("Please set the GOOGLE_API_KEY environment variable.")


In [ ]:
# Creating a document manually
from langchain_core.documents import Document

 # Create a simple document
doc = Document(
  page_content = "Langchain is a framework for building applications with LLMs",
  metadata = {
   "source": "https://www.langchain.com",
   "author": "Langchain Team",
   "date": "2024-06-01"
  }
 )
# Access the conten
print("Content:")
print(doc.page_content)
print("\nMetadata:")
print(doc.metadata)
print("\nSource")
print(doc.metadata["source"])


In [ ]:
# Creating multiple documents manually like a mini knowledge base
documents =[
  Document(
    page_content = "Python is a high level programming language",
    metadata = {
      "category":"programming","difficulty":"beginner"
      }
  ),
  Document(
    page_content ="Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data",
    metadata = {
      "category":"AI","difficulty":"intermediate"
      }
  ),
  Document(
    page_content = "RAG combines retreival and generation for better LLM outputs",
    metadata = {
      "category":"AI","difficulty":"advanced"
      }
  )
]

# Print all documents
for i,doc in enumerate(documents,1):
  print(f"\nDocument {i}:")
  print("Content:", doc.page_content)
  print("Category:", doc.metadata["category"])
  print("Difficulty:", doc.metadata["difficulty"])

In [ ]:
# Simple LCEL example: String transformation using LCEL
from langchain_core.runnables import RunnableLambda

#Create a simple transformation function
def uppercase(text:str)->str:
  """Convert text to uppercase."""
  print(f" Step 1: uppercase -> {text.upper()}")
  return text.upper()

def add_prefix(text:str)->str:
  """Add a prefix to the text."""
  result =f"AI:{text}"
  print(f" Step 2: add_prefix -> {result}")
  return result

def add_emoji(text:str) -> str:
  """Add an emoji to the text."""
  result = f"{text} 🚀"
  print(f" Step 3: add_emoji -> {result}")
  return result

#Create runnablees -> components that can be chained
uppercase_runnable = RunnableLambda(uppercase)
add_prefix_runnable = RunnableLambda(add_prefix)
add_emoji_runnable = RunnableLambda(add_emoji)

#Build the chain using LCEL
chain = uppercase_runnable | add_prefix_runnable | add_emoji_runnable

#Execute the chain using lcel
result = chain.invoke("Hello, Langchain!")
print(f"\nFinal Result: {result}")

In [ ]:
# First LLM call using the langchain
from langchain_openai import ChatOpenAI
# Initialize the LLM with a specific model
llm = ChatOpenAI(model="gpt-5-nano-2025-08-07", temperature=0)

# Make a call to the LLM with a simple prompt
response = llm.invoke("What is Langchain in one sentence?")
print(response.content)

In [ ]:
# The response is an AI message object with metadata
print(type(response))
print("\nContent:",response.content)
print("\nResponse Metadata:",response.response_metadata)

# Access specific metadata fields
if 'token_usage' in response.response_metadata:
  usage = response.response_metadata['token_usage']
  print("\nToken Usage:")
  print(f" Prompt: {usage.get('prompt_tokens', 'N/A')}")
  print(f" Completion: {usage.get('completion_tokens', 'N/A')}")
  print(f" Total: {usage.get('total_tokens', 'N/A')}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a simple prompt template
prompt_template = ChatPromptTemplate.from_template("Explain {topic} in one sentence for beginners.")
#View the prompt structure
print(prompt_template.messages[0].prompt.template)

In [ ]:
# Bild a chain: Prompt -> LLM
from langchain_core.output_parsers import StrOutputParser

# Components
# 1.prompt Formats the input
# 2. llm: Generates the response
# 3 StrOutputParser: Extract the text from the LLM response

chain = prompt_template | llm | StrOutputParser()

#Use the chain with different topics
topics =["Machine learning","ARTIFICIAL intelligence","generative ai","ai vs machine learning"]

for topic in topics:
  print(f"\n{'='*60}")
  print(f"Topic: {topic.upper()}")
  print('='*60)
  #Invoke the chain with the topic
  explanation = chain.invoke({"topic":topic})
  print(f"Explanation: {explanation}")

In [ ]:
# Batch process multiple topics at once
topics_batch = [
  {"topic":"RAG"},
  {"topic":"Langchain expression language"},
  {"topic":"Langchain agents"}
]

results = chain.batch(topics_batch)

# Print results
for i,(input_dict,result) in enumerate(zip(topics_batch,results),1):
  print(f"\n{i}.{input_dict['topic'].upper()}:")
  print(f" {result[:100]}...")


In [ ]:
# Q&A system without langchain
from openai import OpenAI
import PyPDF2
import faiss
import numpy as np

client = OpenAI()

# 1. Load PDF manually
def load_pdf(file_path):
  reader = PyPDF2.PdfReader(file_path)
  text = ""
  for page_num,page in enumerate(reader.pages,start=1):
    print(f"Page {page_num}:")
    #print(f"{page.extract_text()[:50]}")
    text += page.extract_text()
  return text

#2 Split the text manually
def split_text(text, chunk_size=500):
  """Split text into chunks of specified size."""
  chunks = []
  for i in range(0,len(text),chunk_size):
    chunk = text[i:i+chunk_size]
    chunks.append(chunk)
  return chunks

#3 Create embeddings manually
def create_embeddings(chunks):
  embeddings =[]
  for chunk in chunks:
    response = client.embeddings.create(
      input=chunk,
      model="text-embedding-ada-002"
    )
    embeddings.append(response.data[0].embedding)
  return np.array(embeddings)

#4 Create vector store manually
def create_vector_store(embeddings):
  dimension = embeddings.shape[1]
  index = faiss.IndexFlatL2(dimension)
  index.add(embeddings)
  return index

#5 Search manuallly
def search(query,index,chunks):
  query_embedding = client.embeddings.create(
    input=query,
    model="text-embedding-ada-002"
  ).data[0].embedding

  distances,indices = index.search(
    np.array([query_embedding]),k=3
  )

  return [chunks[i] for i in indices[0]]

# 6 Generate answer manually
def generate_answer(query,context):
  prompt = f"Context: {context}\n\nQuestion:{query}\n\nAnswer:"
  response = client.chat.completions.create(
    model="gpt-5-nano-2025-08-07",
    messages=[{"role":"user","content":prompt}]
  )

  return response.choices[0].message.content

#Use it:
text = load_pdf('./Content/attention.pdf')
chunks = split_text(text)
print(chunks)
embeddings = create_embeddings(chunks)
index = create_vector_store(embeddings)
relevant_chunks = search("What is RAG?",index,chunks)
answer = generate_answer("What is RAG"," ".join(relevant_chunks))
print(answer)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

docs = PyPDFLoader("./Content/attention.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000).split_documents(docs)
vectorstore = FAISS.from_documents(chunks,OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

#Build RAG chain
prompt = ChatPromptTemplate.from_template(
  "Context:{context}\n\nQuestion:{question}\n\nAnswer:"
)
chain =(
  {"context":retriever,"question":RunnablePassthrough()}
  | prompt | ChatOpenAI(model="gpt-3.5-turbo") | StrOutputParser()
)

answer = chain.invoke("What is RAG?")
print(answer)